#
<h1 align="center">INTERNET OF THINGS</h1>
<h4 align="center">Mobin khatib(11114435)- Ali Edareh Heidarabadi(11036028)</h4>
<h4 align="center">Politecnico di Milano, Spring 2026</h4>

<h4 align="center">Challenge 2 - Part 2 - Exercise</h4>




This script calculates the total communication energy consumed by a
battery-powered sensor and actuator over 1 hour using CoAP and MQTT.


# PART 1: CONSTANTS & UNIT CONVERSIONS


In [15]:
# Energy cost per bit (in nanojoules, nJ)
E_TX_NJ_PER_BIT = 50
E_RX_NJ_PER_BIT = 58

# Convert energy cost to microjoules (uJ) per byte
TX_UJ_PER_BYTE = (E_TX_NJ_PER_BIT * 8) / 1000.0  # 0.4 uJ/B
RX_UJ_PER_BYTE = (E_RX_NJ_PER_BIT * 8) / 1000.0  # 0.464 uJ/B

# System timing characteristics
MEASUREMENT_INTERVAL_MIN = 2
CYCLES_PER_HOUR = 60 // MEASUREMENT_INTERVAL_MIN  # 30 cycles per hour


# PART 2: EQ1a - CoAP (Direct Communication)


In [16]:
coap_tx_bytes_per_cycle = 70 + 15
coap_rx_bytes_per_cycle = 70 + 15

coap_cycle_uJ = (coap_tx_bytes_per_cycle * TX_UJ_PER_BYTE) + \
                (coap_rx_bytes_per_cycle * RX_UJ_PER_BYTE)

coap_total_uJ = CYCLES_PER_HOUR * coap_cycle_uJ

# PART 3: EQ1b - MQTT (Via Gateway)

In [17]:
# 1. Sensor Energy (1 Connect at start, then 30 Publishes)
sensor_setup_uJ = (60 * TX_UJ_PER_BYTE) + (50 * RX_UJ_PER_BYTE)
sensor_cycle_uJ = (75 * TX_UJ_PER_BYTE) + (50 * RX_UJ_PER_BYTE)
sensor_total_uJ_eq1 = sensor_setup_uJ + (CYCLES_PER_HOUR * sensor_cycle_uJ)

# 2. Actuator Energy (1 Connect & 1 Subscribe at start, then 30 Publishes received)
actuator_setup_uJ = ((60 + 65) * TX_UJ_PER_BYTE) + ((50 + 55) * RX_UJ_PER_BYTE)
actuator_cycle_uJ = (50 * TX_UJ_PER_BYTE) + (75 * RX_UJ_PER_BYTE)
actuator_total_uJ_eq1 = actuator_setup_uJ + (CYCLES_PER_HOUR * actuator_cycle_uJ)

# Total MQTT energy for EQ1b
mqtt_eq1_total_uJ = sensor_total_uJ_eq1 + actuator_total_uJ_eq1



# PART 4: EQ2 - Optional Intuition (Actuator wakes every 30 min)

In [18]:
ACTUATOR_WAKES_PER_HOUR = 60 // 30  # 2 wakes per hour

# Sensor energy remains the same as EQ1b
sensor_total_uJ_eq2 = sensor_total_uJ_eq1

# Actuator wakes just 2 times. Each wake cycle:
# TX: CONNECT(60), SUBSCRIBE(65), PUBACK(50), DISCONNECT(60)
# RX: CONNACK(50), SUBACK(55), PUBLISH(75)
actuator_wake_tx_bytes = 60 + 65 + 50 + 60
actuator_wake_rx_bytes = 50 + 55 + 75

actuator_wake_uJ = (actuator_wake_tx_bytes * TX_UJ_PER_BYTE) + \
                   (actuator_wake_rx_bytes * RX_UJ_PER_BYTE)
actuator_total_uJ_eq2 = ACTUATOR_WAKES_PER_HOUR * actuator_wake_uJ

# Total energy for the optimized scenario
mqtt_eq2_total_uJ = sensor_total_uJ_eq2 + actuator_total_uJ_eq2


# PART 5: PRINT RESULTS

In [20]:
print("==================================================")
print(" [PARAMETERS] Energy Cost & Setup")
print("==================================================")
print(f"TX Cost: {TX_UJ_PER_BYTE:.3f} µJ/Byte")
print(f"RX Cost: {RX_UJ_PER_BYTE:.3f} µJ/Byte\n")

print("==================================================")
print(" [EQ1a] CoAP Direct Communication")
print("==================================================")
print(f"Total per hour   : {coap_total_uJ:.2f} µJ\n")

print("==================================================")
print(" [EQ1b] MQTT Via Gateway (Actuator Always Active)")
print("==================================================")
print(f"Sensor Total     : {sensor_total_uJ_eq1:.2f} µJ")
print(f"Actuator Total   : {actuator_total_uJ_eq1:.2f} µJ")
print(f"Overall EQ1b     : {mqtt_eq1_total_uJ:.2f} µJ\n")

print("==================================================")
print(" [EQ2] Protocol Choice Justification")
print("==================================================")
print(f"Scenario: MQTT with Retained Publish (Actuator wakes 2 times/hr)")
print(f"Sensor Total     : {sensor_total_uJ_eq2:.2f} µJ")
print(f"Actuator Total   : {actuator_total_uJ_eq2:.2f} µJ")
print(f"Overall EQ2      : {mqtt_eq2_total_uJ:.2f} µJ\n")
print("Recommendation   : For long-term operation with deep sleep cycles,")
print("                   MQTT is preferred. By leveraging 'Retained Messages'")
print("                   (retain flag = 1), the actuator can sleep independently")
print("                   and only pull the latest state upon waking, significantly")
print("                   reducing energy waste compared to constant polling.")
print("==================================================")

 [PARAMETERS] Energy Cost & Setup
TX Cost: 0.400 µJ/Byte
RX Cost: 0.464 µJ/Byte

 [EQ1a] CoAP Direct Communication
Total per hour   : 2203.20 µJ

 [EQ1b] MQTT Via Gateway (Actuator Always Active)
Sensor Total     : 1643.20 µJ
Actuator Total   : 1742.72 µJ
Overall EQ1b     : 3385.92 µJ

 [EQ2] Protocol Choice Justification
Scenario: MQTT with Retained Publish (Actuator wakes 2 times/hr)
Sensor Total     : 1643.20 µJ
Actuator Total   : 355.04 µJ
Overall EQ2      : 1998.24 µJ

Recommendation   : For long-term operation with deep sleep cycles,
                   MQTT is preferred. By leveraging 'Retained Messages'
                   (retain flag = 1), the actuator can sleep independently
                   and only pull the latest state upon waking, significantly
                   reducing energy waste compared to constant polling.


Protocol: MQTT. > Justification: It uncouples the devices. The sensor publishes every 2 mins with Retain = True (overwriting old data). The actuator wakes every 30 mins, connects with Clean Session = True, subscribes to receive only the single latest retained temperature, and disconnects, minimizing RX energy waste.